# 04_glove_fasttext: FastText Subwords on Gutenberg Corpus
    
This notebook trains FastText and Word2Vec models on Gutenberg's *Alice in Wonderland* to demonstrate how subword n-grams resolve Out-of-Vocabulary (OOV) queries.


In [1]:
import re
import nltk
from gensim.models import FastText, Word2Vec

# Load sentences
nltk.download('gutenberg', quiet=True)
from nltk.corpus import gutenberg
sentences_raw = gutenberg.sents('carroll-alice.txt')

cleaned_sentences = []
for s in sentences_raw:
    words = [w.lower() for w in s if re.match(r"^\w+$", w)]
    if 5 < len(words) < 35:
        cleaned_sentences.append(words)

train_sentences = cleaned_sentences[:500]
print(f"Prepared {len(train_sentences)} sentences.")


Prepared 500 sentences.


### Output Explanation: Data Preparation
- **Sentences**: Slices 500 cleaned sentences from the *Alice in Wonderland* dataset to feed models.


In [2]:
# Train Word2Vec
w2v = Word2Vec(train_sentences, vector_size=10, window=3, min_count=2, epochs=20)

# Train FastText with subwords bounds min_n=3, max_n=6
ft = FastText(train_sentences, vector_size=10, window=3, min_count=2, min_n=3, max_n=6, epochs=20)

print("Vocab Sample (Word2Vec):", list(w2v.wv.key_to_index.keys())[:8])


Vocab Sample (Word2Vec): ['the', 'i', 'and', 'to', 'it', 'a', 'she', 'you']


### Output Explanation: Training CBOW vs Subwords
- **Vocab Index**: Lists key words present in the dictionary. Next, we will test lookups on unseen words.


In [3]:
# OOV word query (e.g. 'alicean' - not in vocabulary)
try:
    vector = w2v.wv["alicean"]
    print("Word2Vec lookup succeeded.")
except KeyError:
    print("[Word2Vec Result]: KeyError! Word 'alicean' is out of vocabulary!")

# FastText resolves this using character n-gram offsets
ft_vector = ft.wv["alicean"]
print("\nFastText Vector for 'alicean' (first 5 dimensions):\n", ft_vector[:5])

sim_score = ft.wv.similarity("alice", "alicean")
print(f"\nFastText Cosine Similarity ('alice' vs 'alicean'): {sim_score:.4f}")


[Word2Vec Result]: KeyError! Word 'alicean' is out of vocabulary!

FastText Vector for 'alicean' (first 5 dimensions):
 [ 0.26535594 -0.02521512  0.18308373 -0.7056187   0.19021201]

FastText Cosine Similarity ('alice' vs 'alicean'): 0.9998


### Output Explanation: Out-of-Vocabulary Recovery
- **Word2Vec Failure**: Querying `"alicean"` triggers a `KeyError` because static models cannot resolve words missing from their dictionary.
- **FastText Success**: FastText decomposes `"alicean"` into character n-grams and sums their representations to synthesize a vector, resolving the query with a high semantic similarity to `"alice"`.
